In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "bcosgnn").is_dir():
            return p
    raise RuntimeError("Could not locate repo root (pyproject.toml + bcosgnn/).")

project_root = find_repo_root(current_dir)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


# MolHIV: B-COS GINE vs Vanilla GINE
This notebook trains and compares **B-COS GINE** and **Vanilla GINE** on MolHIV using **3 seeds**.

Reported metrics for each model:
- Test ROC-AUC (mean ± std across seeds)
- Best validation epoch (mean ± std across seeds)

In [2]:
import copy
import random
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.nn.aggr import SumAggregation

from bcos.modules import BcosLinear

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.9.1
CUDA available: False


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class TrainConfig:
    hidden_dim: int = 128
    num_layers: int = 4
    dropout: float = 0.2
    b: float = 2.0
    max_out: int = 1
    batch_size: int = 64
    max_epochs: int = 120
    lr: float = 1e-3
    weight_decay: float = 1e-5
    early_stop_patience: int = 20

cfg = TrainConfig()
SEEDS = (0, 1, 2)
TEST_SIZE = 0.2
VAL_SIZE = 0.1
DATASET_ROOT = str(project_root / "data" / "MolHIV")

print(cfg)

TrainConfig(hidden_dim=128, num_layers=4, dropout=0.2, b=2.0, max_out=1, batch_size=64, max_epochs=120, lr=0.001, weight_decay=1e-05, early_stop_patience=20)


In [4]:
def _as_binary_label(y_tensor: torch.Tensor) -> int:
    y = float(y_tensor.view(-1)[0].item())
    return int(y > 0.0)

def load_molhiv_dataset(root: str):
    dataset = MoleculeNet(root=root, name="HIV")
    filtered = []
    for data in dataset:
        y = data.y.view(-1)[0]
        if torch.isfinite(y):
            data.y = torch.tensor([_as_binary_label(data.y)], dtype=torch.long)
            filtered.append(data)
    return filtered

def stratified_split_indices(labels: np.ndarray, seed: int, test_size: float, val_size: float):
    all_idx = np.arange(len(labels))
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(sss_test.split(all_idx, labels))

    labels_trainval = labels[trainval_idx]
    rel_val_size = val_size / (1.0 - test_size)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=rel_val_size, random_state=seed)
    train_rel, val_rel = next(sss_val.split(np.zeros_like(labels_trainval), labels_trainval))

    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel]
    return train_idx, val_idx, test_idx

dataset = load_molhiv_dataset(DATASET_ROOT)
labels = np.array([int(d.y.item()) for d in dataset], dtype=np.int64)

print(f"Loaded MolHIV samples: {len(dataset)}")
print(f"Positive ratio: {labels.mean():.4f}")

Processing...
Processing...
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/torch_geometric/data/dataset.py:265: UserWarning: Skipping molecule 'O=C1O[Al]23(OC1=O)(OC(=O)C(=O)O2)OC(=O)C(=O)O3' since it resulted in zero atoms
  self.process()
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/torch_geometric/data/dataset.py:265: UserWarning: Skipping molecule 'O=C1O[Al]23(OC1=O)(OC(=O)C(=O)O2)OC(=O)C(=O)O3' since it resulted in zero atoms
  self.process()
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/torch_geometric/data/dataset.py:265: UserWarning: Skipping molecule 'Cc1ccc([B-2]2(c3ccc(C)cc3)=NCCO2)cc1' since it resulted in zero atoms
  self.process()
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/torch_geometric/data/dataset.py:265: UserWarning: Skipping molecule 'Cc1ccc([B-2

Loaded MolHIV samples: 41120
Positive ratio: 0.0351


In [5]:
class BCosGINE(nn.Module):
    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 4,
        num_classes: int = 1,
        b: float = 2.0,
        max_out: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out)
        self.convs = nn.ModuleList([
            GINEConv(BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out), train_eps=False)
            for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers)])
        self.agg = SumAggregation()
        self.dropout = nn.Dropout(dropout)
        self.head = BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out)

    def forward(self, x, edge_index, edge_attr, batch):
        x = x.float()
        edge_attr = edge_attr.float()
        x = self.lin_node(x)
        edge_attr = self.lin_edge(edge_attr)
        for conv, bn in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
        g = self.agg(x, batch)
        g = self.dropout(g)
        return self.head(g).view(-1)

class VanillaGINE(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.2):
        super().__init__()
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp, train_eps=False))
            self.norms.append(nn.BatchNorm1d(hidden_dim))
        self.agg = SumAggregation()
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch):
        x = x.float()
        edge_attr = edge_attr.float()
        x = self.node_proj(x)
        edge_attr = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
        g = self.agg(x, batch)
        g = self.dropout(g)
        return self.head(g).view(-1)

In [6]:
def compute_roc_auc_from_logits(logits: List[float], labels: List[int]) -> float:
    y_true = np.asarray(labels, dtype=np.int64)
    y_score = torch.sigmoid(torch.tensor(logits)).numpy()
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def run_epoch(model, loader, optimizer, device):
    is_train = optimizer is not None
    model.train(mode=is_train)
    total_loss = 0.0
    logits_all, y_all = [], []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        y = batch.y.view(-1).float()
        loss = F.binary_cross_entropy_with_logits(logits, y)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += float(loss.item()) * y.size(0)
        logits_all.extend(logits.detach().cpu().tolist())
        y_all.extend(batch.y.view(-1).detach().cpu().tolist())

    avg_loss = total_loss / max(1, len(y_all))
    roc_auc = compute_roc_auc_from_logits(logits_all, y_all)
    return avg_loss, roc_auc

def train_one_seed(model_ctor, dataset, train_idx, val_idx, test_idx, cfg: TrainConfig, seed: int, device: torch.device):
    set_seed(seed)
    sample = dataset[0]
    node_dim = int(sample.x.size(-1))
    edge_dim = int(sample.edge_attr.size(-1))

    model = model_ctor(node_dim=node_dim, edge_dim=edge_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader([dataset[i] for i in test_idx], batch_size=cfg.batch_size, shuffle=False)

    best_state = None
    best_val_auc = -float("inf")
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(1, cfg.max_epochs + 1):
        train_loss, train_auc = run_epoch(model, train_loader, optimizer, device)
        val_loss, val_auc = run_epoch(model, val_loader, optimizer=None, device=device)

        if np.isnan(val_auc):
            val_auc = -float("inf")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Seed {seed} | Epoch {epoch:03d} | train_loss={train_loss:.4f} train_auc={train_auc:.4f} "
                f"| val_loss={val_loss:.4f} val_auc={val_auc:.4f}"
            )

        if bad_epochs >= cfg.early_stop_patience:
            print(f"Seed {seed}: early stop at epoch {epoch} (best val epoch={best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_auc = run_epoch(model, test_loader, optimizer=None, device=device)
    return {
        "seed": int(seed),
        "best_val_epoch": int(best_epoch),
        "best_val_roc_auc": float(best_val_auc),
        "test_roc_auc": float(test_auc),
        "test_loss": float(test_loss),
    }

def summarize_seed_results(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    summary = pd.DataFrame(
        [
            {
                "model": model_name,
                "n_seeds": int(len(df)),
                "test_roc_auc_mean": float(df["test_roc_auc"].mean()),
                "test_roc_auc_std": float(df["test_roc_auc"].std(ddof=1)) if len(df) > 1 else 0.0,
                "best_val_epoch_mean": float(df["best_val_epoch"].mean()),
                "best_val_epoch_std": float(df["best_val_epoch"].std(ddof=1)) if len(df) > 1 else 0.0,
            }
        ]
    )
    return summary

def run_three_seed_experiment(model_name: str, model_ctor, dataset, labels, cfg: TrainConfig, seeds: Iterable[int]):
    device = get_device()
    print(f"\nRunning {model_name} on device: {device}")
    results = []

    for seed in seeds:
        train_idx, val_idx, test_idx = stratified_split_indices(
            labels=labels,
            seed=int(seed),
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
        )
        out = train_one_seed(
            model_ctor=model_ctor,
            dataset=dataset,
            train_idx=train_idx,
            val_idx=val_idx,
            test_idx=test_idx,
            cfg=cfg,
            seed=int(seed),
            device=device,
        )
        results.append(out)
        print(
            f"{model_name} | Seed {seed} | test_auc={out['test_roc_auc']:.4f} "
            f"| best_val_epoch={out['best_val_epoch']} | best_val_auc={out['best_val_roc_auc']:.4f}"
        )

    df = pd.DataFrame(results).sort_values("seed").reset_index(drop=True)
    summary = summarize_seed_results(df, model_name=model_name)
    return df, summary

## B-COS GINE (3 seeds)

In [7]:
bcos_ctor = lambda node_dim, edge_dim: BCosGINE(
    node_dim=node_dim,
    edge_dim=edge_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
    b=cfg.b,
    max_out=cfg.max_out,
)

bcos_runs, bcos_summary = run_three_seed_experiment(
    model_name="B-COS GINE",
    model_ctor=bcos_ctor,
    dataset=dataset,
    labels=labels,
    cfg=cfg,
    seeds=SEEDS,
)

display(bcos_runs)
display(bcos_summary)


Running B-COS GINE on device: cpu
Seed 0 | Epoch 001 | train_loss=0.2375 train_auc=0.5738 | val_loss=0.2220 val_auc=0.5716
Seed 0 | Epoch 001 | train_loss=0.2375 train_auc=0.5738 | val_loss=0.2220 val_auc=0.5716
Seed 0 | Epoch 010 | train_loss=0.1463 train_auc=0.7391 | val_loss=0.1585 val_auc=0.6971
Seed 0 | Epoch 010 | train_loss=0.1463 train_auc=0.7391 | val_loss=0.1585 val_auc=0.6971
Seed 0 | Epoch 020 | train_loss=0.1265 train_auc=0.8036 | val_loss=0.1526 val_auc=0.7468
Seed 0 | Epoch 020 | train_loss=0.1265 train_auc=0.8036 | val_loss=0.1526 val_auc=0.7468
Seed 0 | Epoch 030 | train_loss=0.1151 train_auc=0.8376 | val_loss=0.1623 val_auc=0.7342
Seed 0 | Epoch 030 | train_loss=0.1151 train_auc=0.8376 | val_loss=0.1623 val_auc=0.7342
Seed 0 | Epoch 040 | train_loss=0.1064 train_auc=0.8728 | val_loss=0.1547 val_auc=0.7576
Seed 0 | Epoch 040 | train_loss=0.1064 train_auc=0.8728 | val_loss=0.1547 val_auc=0.7576
Seed 0 | Epoch 050 | train_loss=0.0968 train_auc=0.9023 | val_loss=0.1757 v

,seed,best_val_epoch,best_val_roc_auc,test_roc_auc,test_loss
0,0,45,0.771418,0.780226,0.140829
1,1,58,0.830985,0.814060,0.137611
2,2,114,0.863157,0.826156,0.158553


,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,B-COS GINE,3,0.806814,0.023807,72.333333,36.665151


## Vanilla GINE (3 seeds)

In [8]:
vanilla_ctor = lambda node_dim, edge_dim: VanillaGINE(
    node_dim=node_dim,
    edge_dim=edge_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
)

vanilla_runs, vanilla_summary = run_three_seed_experiment(
    model_name="Vanilla GINE",
    model_ctor=vanilla_ctor,
    dataset=dataset,
    labels=labels,
    cfg=cfg,
    seeds=SEEDS,
)

display(vanilla_runs)
display(vanilla_summary)


Running Vanilla GINE on device: cpu
Seed 0 | Epoch 001 | train_loss=0.3188 train_auc=0.4479 | val_loss=0.2276 val_auc=0.4076
Seed 0 | Epoch 001 | train_loss=0.3188 train_auc=0.4479 | val_loss=0.2276 val_auc=0.4076
Seed 0 | Epoch 010 | train_loss=0.1461 train_auc=0.6786 | val_loss=0.2189 val_auc=0.5468
Seed 0 | Epoch 010 | train_loss=0.1461 train_auc=0.6786 | val_loss=0.2189 val_auc=0.5468
Seed 0 | Epoch 020 | train_loss=0.1271 train_auc=0.7607 | val_loss=0.1395 val_auc=0.7144
Seed 0 | Epoch 020 | train_loss=0.1271 train_auc=0.7607 | val_loss=0.1395 val_auc=0.7144
Seed 0 | Epoch 030 | train_loss=0.1191 train_auc=0.7914 | val_loss=0.2076 val_auc=0.7088
Seed 0 | Epoch 030 | train_loss=0.1191 train_auc=0.7914 | val_loss=0.2076 val_auc=0.7088
Seed 0 | Epoch 040 | train_loss=0.1119 train_auc=0.8307 | val_loss=0.1333 val_auc=0.7464
Seed 0 | Epoch 040 | train_loss=0.1119 train_auc=0.8307 | val_loss=0.1333 val_auc=0.7464
Seed 0 | Epoch 050 | train_loss=0.1093 train_auc=0.8434 | val_loss=0.1261

,seed,best_val_epoch,best_val_roc_auc,test_roc_auc,test_loss
0,0,63,0.782346,0.800641,0.113336
1,1,114,0.835505,0.780884,0.131119
2,2,99,0.842812,0.786474,0.122661


,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,Vanilla GINE,3,0.789333,0.010184,92.0,26.210685


## Comparative Report

In [9]:
comparison = pd.concat([bcos_summary, vanilla_summary], ignore_index=True)

comparison = comparison[[
    "model",
    "n_seeds",
    "test_roc_auc_mean",
    "test_roc_auc_std",
    "best_val_epoch_mean",
    "best_val_epoch_std",
]]

display(comparison)

print("\nCompact report:")
for _, row in comparison.iterrows():
    print(
        f"{row['model']}: test ROC-AUC = {row['test_roc_auc_mean']:.4f} ± {row['test_roc_auc_std']:.4f}; "
        f"best val epoch = {row['best_val_epoch_mean']:.2f} ± {row['best_val_epoch_std']:.2f}"
    )

,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,B-COS GINE,3,0.806814,0.023807,72.333333,36.665151
1,Vanilla GINE,3,0.789333,0.010184,92.000000,26.210685



Compact report:
B-COS GINE: test ROC-AUC = 0.8068 ± 0.0238; best val epoch = 72.33 ± 36.67
Vanilla GINE: test ROC-AUC = 0.7893 ± 0.0102; best val epoch = 92.00 ± 26.21
